# Ứng dụng thực tế

## Mục tiêu
Áp dụng frequent itemset cho event log. Ta tạo một bộ log mô phỏng, chạy các thuật toán trong project, rồi đọc output để diễn giải các tổ hợp sự kiện xuất hiện thường xuyên.


In [ ]:
function find_project_root(start::AbstractString)
    current = abspath(start)

    while true
        if isfile(joinpath(current, "src", "main.jl"))
            return current
        end

        parent = dirname(current)
        parent == current && error("Không tìm thấy thư mục dự án chứa src/main.jl")
        current = parent
    end
end

PROJECT_ROOT = find_project_root(pwd())
SRC_DIR = joinpath(PROJECT_ROOT, "src")
NOTEBOOK_DIR = joinpath(PROJECT_ROOT, "notebook")
DATA_DIR = joinpath(PROJECT_ROOT, "data", "eventlog_demo")
OUTPUT_DIR = joinpath(PROJECT_ROOT, "data", "eventlog_output")

mkpath(DATA_DIR)
mkpath(OUTPUT_DIR)

println("PROJECT_ROOT = ", PROJECT_ROOT)
println("SRC_DIR      = ", SRC_DIR)
println("DATA_DIR     = ", DATA_DIR)
println("OUTPUT_DIR   = ", OUTPUT_DIR)


In [ ]:
event_map = Dict(
    1 => "AUTH_SUCCESS",
    2 => "AUTH_FAILURE",
    3 => "PRIV_ESCALATION_ATTEMPT",
    4 => "MULTIPLE_404",
    5 => "SUDO_COMMAND",
    6 => "FILE_PERMISSION_DENIED",
    7 => "SERVICE_RESTART",
    8 => "OUTBOUND_CONN_NEW_IP",
    9 => "HIGH_CPU",
    10 => "BACKUP_COMPLETED",
    11 => "CONFIG_CHANGE",
    12 => "DATABASE_ERROR",
)

transactions = [
    [1, 10],
    [1, 10, 7],
    [2, 3, 5, 6],
    [2, 3, 5, 8, 11],
    [1, 10, 9],
    [2, 3, 5, 6, 8],
    [7, 9, 12],
    [2, 3, 5, 11],
    [1, 10, 11],
    [2, 3, 5, 6],
    [7, 9, 12, 6],
    [2, 3, 5, 8],
    [1, 10],
    [2, 3, 5, 6, 11],
    [7, 9, 12],
    [1, 10, 4],
    [2, 3, 5, 6, 8, 11],
    [1, 10, 7],
    [2, 3, 5, 6],
    [7, 9, 12, 11],
]

input_file = joinpath(DATA_DIR, "event_log_demo.txt")
open(input_file, "w") do io
    for tr in transactions
        println(io, join(sort(unique(tr)), " "))
    end
end

map_file = joinpath(DATA_DIR, "event_id_mapping.txt")
open(map_file, "w") do io
    for id in sort(collect(keys(event_map)))
        println(io, string(id, ": ", event_map[id]))
    end
end

println("Đã tạo input: ", input_file)
println("Đã tạo mapping: ", map_file)
println("Số giao dịch: ", length(transactions))


## Quy trình chạy thực nghiệm bằng CLI

### Bước 1. Chuẩn bị tham số
- Notebook này gọi lại `src/main.jl` bằng chính Julia đang chạy notebook.
- Điều chỉnh `MINSUP` trong khoảng `[0, 1]` nếu muốn thử ngưỡng khác.
- Dữ liệu demo được lưu trong `data/eventlog_demo`.

### Bước 2. Chạy các chế độ
- `-a`: chạy từng thuật toán (`classic`, `projection`, `adjacency`).
- `-c`: so sánh hai thuật toán trên cùng input.
- `-ca`: chạy toàn bộ thuật toán trên cùng input.

### Bước 3. Kiểm tra kết quả
- Output được ghi trong `data/eventlog_output`.


In [ ]:
JULIA_CMD = Base.julia_cmd()
MINSUP = 0.30

function show_cmd(cmd::Cmd)
    return join(map(string, collect(cmd)), " ")
end

function run_cli(args::Vector{String})
    script_path = joinpath(SRC_DIR, "main.jl")
    base_cmd = `$JULIA_CMD --project=$PROJECT_ROOT $script_path $(args...)`
    cmd = Cmd(base_cmd; dir = SRC_DIR)
    println("> ", show_cmd(cmd))
    run(cmd)
    println()
end

for alg in ["classic", "projection", "adjacency"]
    run_cli(["-a", alg, input_file, OUTPUT_DIR, string(MINSUP)])
end

run_cli(["-c", "classic", "projection", input_file, OUTPUT_DIR, string(MINSUP)])
run_cli(["-ca", input_file, OUTPUT_DIR, string(MINSUP)])


In [ ]:
using Printf

function minsup_label(minsup::Float64)
    percent = round(minsup * 100; digits = 2)
    return isinteger(percent) ? "$(Int(percent))%" : "$(percent)%"
end

function parse_output(path::AbstractString)
    patterns = NamedTuple{(:items, :support), Tuple{Vector{Int}, Int}}[]

    for line in eachline(path)
        stripped = strip(line)
        isempty(stripped) && continue

        parts = split(stripped, "#SUP:"; limit = 2)
        length(parts) == 2 || continue

        left = strip(parts[1])
        items = isempty(left) ? Int[] : parse.(Int, split(left))
        support = parse(Int, strip(parts[2]))
        push!(patterns, (items = items, support = support))
    end

    return patterns
end

decode_pattern(items::Vector{Int}) = [event_map[i] for i in items]

label = minsup_label(MINSUP)
base = splitext(basename(input_file))[1]

all_patterns = Dict{String, Vector{NamedTuple{(:items, :support), Tuple{Vector{Int}, Int}}}}()

for alg in ["classic", "projection", "adjacency"]
    out_file = joinpath(OUTPUT_DIR, "local_$(alg)_$(base)_$(label).txt")

    if !isfile(out_file)
        println("[CẢNH BÁO] Chưa có file output: ", out_file)
        continue
    end

    patterns = parse_output(out_file)
    all_patterns[alg] = patterns

    println("\n", repeat("=", 70))
    println("Thuật toán: ", alg, " | số mẫu: ", length(patterns))
    println(repeat("=", 70))

    ranked = sort(patterns; by = p -> (-p.support, -length(p.items), join(string.(p.items), ",")))
    for p in Iterators.take(ranked, 12)
        println(@sprintf("support=%2d | ids=%s | sự kiện=%s", p.support, repr(p.items), repr(decode_pattern(p.items))))
    end
end


In [ ]:
suspicious_ids = Set([2, 3, 5, 6, 8, 11, 12])

classic_patterns = get(all_patterns, "classic", NamedTuple{(:items, :support), Tuple{Vector{Int}, Int}}[])
suspicious_patterns = NamedTuple{(:pattern, :overlap), Tuple{NamedTuple{(:items, :support), Tuple{Vector{Int}, Int}}, Vector{Int}}}[]

for p in classic_patterns
    overlap = sort(collect(intersect(Set(p.items), suspicious_ids)))
    if length(overlap) >= 2
        push!(suspicious_patterns, (pattern = p, overlap = overlap))
    end
end

ranked_suspicious = sort(suspicious_patterns; by = x -> (-x.pattern.support, -length(x.pattern.items), join(string.(x.pattern.items), ",")))

println("Các tổ hợp thường xuyên đáng ngờ nhất (classic):")
for entry in Iterators.take(ranked_suspicious, 10)
    println(@sprintf("support=%2d | pattern=%s | suspicious_part=%s", entry.pattern.support, repr(decode_pattern(entry.pattern.items)), repr(decode_pattern(entry.overlap))))
end

println("\nDiễn giải:")
println("- Nếu tổ hợp AUTH_FAILURE + PRIV_ESCALATION_ATTEMPT + SUDO_COMMAND xuất hiện lặp lại, đó có thể là mẫu tấn công leo thang đặc quyền.")
println("- Nếu FILE_PERMISSION_DENIED / OUTBOUND_CONN_NEW_IP / CONFIG_CHANGE đi cùng các sự kiện trên, cần ưu tiên điều tra.")
println("- Frequent itemset giúp tìm mẫu lặp lại; để bắt anomaly hiếm, cần giảm minsup hoặc kết hợp detector theo thời gian.")


## Kết quả và ý nghĩa

### 1. Mẫu đồng xuất hiện trong log
Frequent itemset cho thấy những nhóm sự kiện lặp lại nhiều lần trong hệ thống. Đây là cơ sở để mô tả hành vi vận hành bình thường hoặc hành vi cần theo dõi.

### 2. Khả năng phát hiện lỗi và rủi ro bảo mật
Các mẫu có support cao chứa những sự kiện như `AUTH_FAILURE`, `PRIV_ESCALATION_ATTEMPT`, `SUDO_COMMAND` có thể phản ánh chuỗi hành vi tấn công hoặc cấu hình sai lặp lại.

### 3. Cách dùng trong giám sát thực tế
Có thể dùng các mẫu frequent làm baseline. Khi một pattern có support tăng mạnh theo thời gian, hệ thống có thể phát cảnh báo sớm để kiểm tra.

### 4. Hạn chế
Anomaly hiếm thường không đủ support để trở thành frequent itemset. Vì vậy cần kết hợp thêm các kỹ thuật khác như giảm `minsup`, phân tích theo cửa sổ thời gian, hoặc dùng thêm luật kết hợp.
